<a href="https://colab.research.google.com/github/DrVivasMachado2/awesome-notebookLM-prompts/blob/main/Baloto_Master_AI_v6_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎱 Baloto Master AI **v6.0** — Edición limpia y honesta

**Qué cambió frente a la v5.0** (302 celdas → 14 celdas, 18 motores → 1 motor):

| Módulo | v5.0 | v6.0 |
|---|---|---|
| Datos | CSV manual + OCR con *relleno aleatorio* si fallaba | **Scraper oficial de baloto.com** (Baloto **y** Revancha, con tabla de premios real), caché en Google Drive, OCR solo con validación |
| Motor | 18 clases encadenadas, "confianza 94 %" inventada | 1 clase `BalotoMasterEngineV6` con pesos explícitos y **índice de unicidad** (métrica real) |
| Validación | Ninguna | **Backtesting walk-forward** motor vs. azar + prueba χ² de uniformidad |
| Finanzas | — | **Valor esperado real** por tiquete usando premios oficiales por categoría |
| Instalación | Tesseract se instalaba 12 veces | Una sola vez, solo si falta |

> ⚠️ **Lo honesto primero.** Cada sorteo de Baloto es independiente: la probabilidad del premio mayor es **1 en 15.401.568** (5/43 + 1/16) para *cualquier* combinación, la elija un humano, una red neuronal o un dado. Lo que esta app sí hace bien: traer datos oficiales, describirlos con rigor, calcular cuánto vale realmente un tiquete, evitar combinaciones *populares* (para no repartir el premio si llega) y llevar el control de tus jugadas.

**Orden de ejecución:** celdas 1 → 2 → 3 (datos) → 4–6 (análisis) → 7–8 (motor y backtest) → 9–10 (OCR y mis jugadas, opcionales) → 11 (panel) → 12 (exportar). Si el entorno se reinicia, basta correr 1 → 3 y luego lo que necesites.


In [ ]:
# @title 1️⃣ Configuración e instalación (ejecutar una sola vez por sesión)
# ─────────────────────────────────────────────────────────────────────────────
#  Detecta el entorno (Colab o local), instala dependencias SOLO si faltan,
#  monta Google Drive (opcional) y define las constantes del juego.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, re, json, math, shutil, subprocess, warnings
from datetime import datetime, timedelta, date
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ── Parámetros editables ────────────────────────────────────────────────────
USAR_DRIVE = True            # True → guarda histórico y jugadas en Google Drive (persisten entre sesiones)
CARPETA_DRIVE = "BalotoMasterAI"   # Carpeta dentro de "Mi unidad"
PRECIO_BALOTO = 5_700        # COP, apuesta sencilla. ⚠️ Verificar en baloto.com (jun-2026: $5.700)
PRECIO_CON_REVANCHA = 7_800  # COP, Baloto + Revancha. ⚠️ Verificar (fuentes 2026 oscilan 7.800–9.200)
IMPUESTO_GANANCIA_OCASIONAL = 0.20   # 20 % de retención sobre premios > 48 UVT (Colombia)

# ── Reglas del juego (Baloto y Revancha comparten formato desde feb-2017) ───
N_BALOTAS = 43               # balotas principales: 1..43
K_ESCOGIDAS = 5              # se escogen 5
N_SUPERBALOTA = 16           # superbalota: 1..16
DIAS_SORTEO = {0: "Lunes", 2: "Miércoles", 5: "Sábado"}   # weekday() de Python
HORA_SORTEO = 23             # 11:00 p.m. hora Colombia

# ── Detección de entorno ────────────────────────────────────────────────────
MODO_TEST_LOCAL = os.environ.get("BALOTO_LOCAL_TEST") == "1"
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def _pip_install(paquetes):
    faltan = []
    for mod, pkg in paquetes:
        try:
            __import__(mod)
        except ImportError:
            faltan.append(pkg)
    if faltan:
        print(f"📦 Instalando: {', '.join(faltan)} …")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=False)

if EN_COLAB:
    _pip_install([("pytesseract", "pytesseract"), ("fpdf", "fpdf2"), ("openpyxl", "openpyxl")])
    if shutil.which("tesseract") is None:
        print("📦 Instalando Tesseract OCR (una sola vez) …")
        subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null",
                       shell=True, check=False)
    print("✅ Tesseract:", shutil.which("tesseract") or "no disponible (el OCR quedará desactivado)")

# ── Carpeta de datos persistente ────────────────────────────────────────────
if EN_COLAB and USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DATA_DIR = Path("/content/drive/MyDrive") / CARPETA_DRIVE
    except Exception as e:  # el usuario canceló el permiso, etc.
        print("⚠️ No se pudo montar Drive:", e, "→ se usará /content (se pierde al reiniciar)")
        DATA_DIR = Path("/content") / CARPETA_DRIVE
elif EN_COLAB:
    DATA_DIR = Path("/content") / CARPETA_DRIVE
else:
    DATA_DIR = Path(os.environ.get("BALOTO_DATA_DIR", Path.cwd() / "BalotoMasterAI_data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVOS = {
    "baloto":   DATA_DIR / "historico_baloto.csv",
    "revancha": DATA_DIR / "historico_revancha.csv",
    "premios":  DATA_DIR / "premios_por_categoria.csv",
    "jugadas":  DATA_DIR / "mis_jugadas.csv",
}

# ── Utilidades compartidas ──────────────────────────────────────────────────
COLS_BALOTAS = ["B1", "B2", "B3", "B4", "B5"]

def proximo_sorteo(ahora: datetime | None = None) -> datetime:
    """Fecha del próximo sorteo (lunes, miércoles o sábado, 11 p.m. hora Colombia, UTC-5)."""
    if ahora is None:
        ahora = datetime.utcnow() - timedelta(hours=5)
    for d in range(0, 8):
        cand = (ahora + timedelta(days=d)).replace(hour=HORA_SORTEO, minute=0, second=0, microsecond=0)
        if cand.weekday() in DIAS_SORTEO and cand > ahora:
            return cand
    raise RuntimeError("No se encontró próximo sorteo")

def fmt_cop(x) -> str:
    """Formato pesos colombianos: $1.234.567"""
    try:
        return "$" + f"{int(round(float(x))):,}".replace(",", ".")
    except (TypeError, ValueError):
        return "—"

def validar_combinacion(balotas, sb) -> bool:
    b = [int(x) for x in balotas]
    return (len(b) == K_ESCOGIDAS and len(set(b)) == K_ESCOGIDAS
            and all(1 <= x <= N_BALOTAS for x in b) and 1 <= int(sb) <= N_SUPERBALOTA)

_ps = proximo_sorteo()
print(f"🌍 Entorno: {'Google Colab' if EN_COLAB else 'Local'} | 📁 Datos en: {DATA_DIR}")
print(f"🎱 Reglas: {K_ESCOGIDAS} de {N_BALOTAS} + superbalota de {N_SUPERBALOTA} "
      f"→ combinaciones posibles: {math.comb(N_BALOTAS, K_ESCOGIDAS) * N_SUPERBALOTA:,}")
print(f"📅 Próximo sorteo: {DIAS_SORTEO[_ps.weekday()]} {_ps:%d-%m-%Y} {_ps:%H:%M} (hora Colombia)")


In [ ]:
# @title 2️⃣ Scraper oficial de baloto.com (Baloto + Revancha + tabla de premios)
# ─────────────────────────────────────────────────────────────────────────────
#  baloto.com publica una página por sorteo:
#     https://baloto.com/resultados-baloto/<N>     y     https://baloto.com/resultados-revancha/<N>
#  con fecha, números, acumulado y la tabla PREMIO / GANADORES por categoría.
#  Este módulo las lee de forma INCREMENTAL (solo baja lo que falta) y guarda en DATA_DIR.
#  Páginas disponibles desde ~sorteo 2.076 (julio 2021) hasta el más reciente.
# ─────────────────────────────────────────────────────────────────────────────
import time
import requests
from concurrent.futures import ThreadPoolExecutor

BASE_URL = "https://baloto.com"
SORTEO_INICIO = 2076          # el primero publicado en la web oficial (los anteriores devuelven la página genérica)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/124.0 Safari/537.36", "Accept-Language": "es-CO,es;q=0.9"}
MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
CATEGORIAS_WEB = ["5 + SB", "5", "4 + SB", "4", "3 + SB", "3", "2 + SB", "0 + SB"]
# Nota: en la web "0 + SB" agrupa los tiquetes que SOLO aciertan la superbalota (0 o 1 balota + SB).

_RE_CABECERA = re.compile(
    r"SORTEO\s+([\d.]+)\s+(\w+)\s+(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})\s+"
    r"ACUMULADO DEL SORTEO:\s*\$\s*([\d.,]+)\s*MILLONES"
    r"(?:\s+MIRA EL VIDEO OFICIAL DEL SORTEO)?\s+"
    r"((?:\d{1,2}\s+){5})(\d{1,2})\s+TOTAL GANADORES\s+([\d.]+)\s+TOTAL PREMIACI[ÓO]N\s+\$\s*([\d.]+)",
    re.IGNORECASE)
_RE_PREMIO = re.compile(r"(5 \+ SB|5|4 \+ SB|4|3 \+ SB|3|2 \+ SB|0 \+ SB)\s+\$\s*([\d.]+)\s+([\d.]+)\s+\$\s*([\d.]+)")

def _a_entero(s: str) -> int:
    return int(re.sub(r"[^\d]", "", s) or 0)

def html_a_texto(html: str) -> str:
    html = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", html, flags=re.S | re.I)
    txt = re.sub(r"<[^>]+>", " ", html)
    return re.sub(r"\s+", " ", txt)

def parsear_detalle(html: str, juego: str) -> dict | None:
    """Convierte la página de detalle de un sorteo en un dict. None si la página no tiene resultado."""
    txt = html_a_texto(html)
    m = _RE_CABECERA.search(txt)
    if not m:
        return None
    sorteo, dia_txt, dd, mes_txt, yyyy, acum, nums, sb, tot_gan, tot_prem = m.groups()
    mes = MESES.get(mes_txt.lower())
    if mes is None:
        return None
    balotas = [int(x) for x in nums.split()]
    fila = {
        "Juego": juego, "Sorteo": _a_entero(sorteo),
        "Fecha": date(int(yyyy), mes, int(dd)).isoformat(), "DiaSemana": dia_txt.capitalize(),
        **{c: v for c, v in zip(COLS_BALOTAS, balotas)}, "SB": int(sb),
        "AcumuladoMillones": float(acum.replace(".", "").replace(",", ".")) if acum.count(",") == 1 and acum.count(".") == 0
                             else float(acum.replace(".", "")),
        "TotalGanadores": _a_entero(tot_gan), "TotalPremiacion": _a_entero(tot_prem),
    }
    # tabla de premios (entre "PREMIO POR GANADOR" y "SB Super balota")
    seg = txt[m.end():]
    fin = seg.find("Super balota")
    seg = seg[:fin] if fin > 0 else seg
    premios = {}
    for cat, prem_tot, gan, prem_ind in _RE_PREMIO.findall(seg):
        premios[cat] = {"PremioTotal": _a_entero(prem_tot), "Ganadores": _a_entero(gan), "PremioPorGanador": _a_entero(prem_ind)}
    fila["Premios"] = premios
    return fila

def descargar_sorteo(sorteo: int, juego: str, intentos: int = 3, pausa: float = 0.8) -> dict | None:
    url = f"{BASE_URL}/resultados-{juego}/{sorteo}"
    for i in range(intentos):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code == 200:
                return parsear_detalle(r.text, juego)
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(pausa * (i + 1))
    return None

def _cargar_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        if "Fecha" in df.columns:
            df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date
        return df
    return pd.DataFrame()

def _filas_a_frames(filas: list[dict]):
    res, prem = [], []
    for f in filas:
        f = dict(f)
        premios = f.pop("Premios", {}) or {}
        res.append(f)
        for cat, d in premios.items():
            prem.append({"Juego": f["Juego"], "Sorteo": f["Sorteo"], "Fecha": f["Fecha"], "Categoria": cat, **d})
    return pd.DataFrame(res), pd.DataFrame(prem)

URLS_SEMILLA = [f"https://raw.githubusercontent.com/DrVivasMachado2/awesome-notebookLM-prompts/{rama}/baloto_v6/cache_datos_para_Drive/"
                for rama in ("main", "baloto-v6")]

def descargar_semilla_github(verbose: bool = True) -> None:
    """Si no hay caché local, baja del repo de GitHub la caché publicada con la v6 (evita re-descargar ~660 sorteos)."""
    for clave in ("baloto", "revancha", "premios"):
        destino = ARCHIVOS[clave]
        if destino.exists():
            continue
        for base in URLS_SEMILLA:
            try:
                r = requests.get(base + destino.name, timeout=30)
                if r.ok and r.content[:5] == b"Juego":
                    destino.write_bytes(r.content)
                    if verbose:
                        print(f"⬇️ Caché inicial descargada de GitHub: {destino.name} ({len(r.content)/1024:,.0f} KB)")
                    break
            except requests.RequestException:
                continue

def actualizar_historico(juego: str = "baloto", hasta: int | None = None, workers: int = 4,
                         lote: int = 8, max_lotes_vacios: int = 2, verbose: bool = True) -> pd.DataFrame:
    """Descarga solo los sorteos que faltan en el CSV local y lo actualiza. juego ∈ {'baloto','revancha'}."""
    assert juego in ("baloto", "revancha")
    if not ARCHIVOS[juego].exists():
        descargar_semilla_github(verbose)
    df_prev = _cargar_csv(ARCHIVOS[juego])
    desde = int(df_prev["Sorteo"].max()) + 1 if not df_prev.empty else SORTEO_INICIO
    nuevos, n, lotes_vacios = [], desde, 0
    t0 = time.time()
    if verbose:
        print(f"🌐 [{juego}] Caché local: {len(df_prev)} sorteos. Buscando desde el #{desde} …")
    while lotes_vacios < max_lotes_vacios and (hasta is None or n <= hasta):
        rango = [s for s in range(n, n + lote) if hasta is None or s <= hasta]
        with ThreadPoolExecutor(max_workers=workers) as ex:
            res = list(ex.map(lambda s: descargar_sorteo(s, juego), rango))
        ok = [r for r in res if r]
        nuevos.extend(ok)
        lotes_vacios = lotes_vacios + 1 if not ok else 0
        n += lote
        if verbose and ok:
            print(f"   ↳ hasta #{max(r['Sorteo'] for r in ok)} ({len(nuevos)} nuevos, {time.time()-t0:,.0f}s)", end="\r")
    df_new, prem_new = _filas_a_frames(nuevos)
    if not df_new.empty:
        df_new["Fecha"] = pd.to_datetime(df_new["Fecha"]).dt.date
        df = pd.concat([df_prev, df_new], ignore_index=True).drop_duplicates("Sorteo").sort_values("Sorteo")
        df.to_csv(ARCHIVOS[juego], index=False)
        prem_prev = _cargar_csv(ARCHIVOS["premios"])
        prem = pd.concat([prem_prev, prem_new], ignore_index=True)
        prem = prem.drop_duplicates(["Juego", "Sorteo", "Categoria"]).sort_values(["Juego", "Sorteo"])
        prem.to_csv(ARCHIVOS["premios"], index=False)
    else:
        df = df_prev
    if verbose:
        ult = df.iloc[-1] if not df.empty else None
        print(f"\n✅ [{juego}] {len(df)} sorteos en total (+{len(df_new)} nuevos). "
              + (f"Último: #{int(ult.Sorteo)} {ult.Fecha} → {[int(x) for x in ult[COLS_BALOTAS]]} + SB {int(ult.SB)}" if ult is not None else ""))
    return df

print("✅ Scraper listo. Usa:  actualizar_historico('baloto')  y  actualizar_historico('revancha')")
print("   La primera vez descarga ~660 sorteos por juego (3–6 min). Después solo baja los nuevos.")


In [ ]:
# @title 3️⃣ Cargar / actualizar datos y validarlos
# ─────────────────────────────────────────────────────────────────────────────
#  ACTUALIZAR_DESDE_WEB = True  → consulta baloto.com y trae los sorteos nuevos.
#  Si no hay internet o prefieres trabajar con la caché, ponlo en False.
#  CSV_LEGADO: tu CSV antiguo (2009-2026). Se puede cargar SOLO como referencia:
#  sus fechas/numeración NO coinciden con la web oficial (p. ej. #2615 figura como
#  2026-06-06 en el CSV, pero oficialmente el #2616 fue el 09-02-2026), así que
#  por defecto NO se mezcla con el histórico oficial.
# ─────────────────────────────────────────────────────────────────────────────
ACTUALIZAR_DESDE_WEB = True
CSV_LEGADO = None   # p.ej. "/content/resultados_baloto_pag1_117.csv"  (solo referencia, no se mezcla)

def validar_historico(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    if df.empty:
        print(f"⚠️ [{nombre}] sin datos.")
        return df
    df = df.copy()
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    ok_rango = df[COLS_BALOTAS].ge(1).all(axis=1) & df[COLS_BALOTAS].le(N_BALOTAS).all(axis=1) \
               & df["SB"].between(1, N_SUPERBALOTA)
    ok_unicos = df[COLS_BALOTAS].nunique(axis=1).eq(K_ESCOGIDAS)
    malos = df[~(ok_rango & ok_unicos)]
    if len(malos):
        print(f"⚠️ [{nombre}] {len(malos)} filas inválidas descartadas:", malos["Sorteo"].tolist()[:10])
    df = df[ok_rango & ok_unicos].drop_duplicates("Sorteo").sort_values("Sorteo").reset_index(drop=True)
    # Ordenar balotas dentro de cada fila (la web ya las publica ordenadas; esto es defensivo)
    df[COLS_BALOTAS] = np.sort(df[COLS_BALOTAS].values, axis=1)
    faltantes = sorted(set(range(df["Sorteo"].min(), df["Sorteo"].max() + 1)) - set(df["Sorteo"]))
    dias = df["Fecha"].dt.weekday.map(lambda w: DIAS_SORTEO.get(w, "Otro")).value_counts().to_dict()
    print(f"✅ [{nombre}] {len(df)} sorteos | #{df.Sorteo.min()} ({df.Fecha.min():%d-%m-%Y}) → "
          f"#{df.Sorteo.max()} ({df.Fecha.max():%d-%m-%Y}) | días: {dias}"
          + (f" | ⚠️ faltan {len(faltantes)} sorteos: {faltantes[:8]}…" if faltantes else ""))
    return df

if ACTUALIZAR_DESDE_WEB:
    df_baloto   = actualizar_historico("baloto")
    df_revancha = actualizar_historico("revancha")
else:
    df_baloto, df_revancha = _cargar_csv(ARCHIVOS["baloto"]), _cargar_csv(ARCHIVOS["revancha"])

df_baloto   = validar_historico(df_baloto, "Baloto")
df_revancha = validar_historico(df_revancha, "Revancha")
df_premios  = _cargar_csv(ARCHIVOS["premios"])
if not df_premios.empty:
    df_premios["Fecha"] = pd.to_datetime(df_premios["Fecha"])

# Histórico combinado: Baloto + Revancha son dos sorteos independientes con las mismas reglas,
# así que para estadística descriptiva se pueden analizar juntos (más muestra) o por separado.
df_hist = pd.concat([df_baloto, df_revancha], ignore_index=True).sort_values(["Fecha", "Juego"]).reset_index(drop=True)

df_legado = None
if CSV_LEGADO and Path(CSV_LEGADO).exists():
    df_legado = pd.read_csv(CSV_LEGADO, parse_dates=["Fecha"])
    print(f"ℹ️ CSV legado cargado solo como referencia: {len(df_legado)} filas ({df_legado.Fecha.min():%Y} → {df_legado.Fecha.max():%Y}).")

if not df_baloto.empty:
    print("\n🗓️ Últimos 5 sorteos de Baloto:")
    print(df_baloto.tail(5)[["Sorteo", "Fecha", "DiaSemana", *COLS_BALOTAS, "SB", "AcumuladoMillones", "TotalGanadores"]]
          .assign(Fecha=lambda d: d.Fecha.dt.strftime("%d-%m-%Y")).to_string(index=False))


In [ ]:
# @title 4️⃣ Análisis estadístico descriptivo (frecuencias, calientes/fríos, par/impar, superbalota)
# ─────────────────────────────────────────────────────────────────────────────
#  Cada hallazgo viene acompañado de la prueba estadística que dice si la
#  desviación observada es "ruido" compatible con un sorteo justo (lo normal) o no.
# ─────────────────────────────────────────────────────────────────────────────
from scipy import stats

VENTANA_RECIENTE = 50   # sorteos recientes para "calientes/fríos"
JUEGO_ANALISIS = "ambos"   # "baloto", "revancha" o "ambos"

def _seleccionar(df_b, df_r, juego):
    return {"baloto": df_b, "revancha": df_r}.get(juego, pd.concat([df_b, df_r], ignore_index=True)).sort_values(["Fecha", "Sorteo"])

def frecuencias(df: pd.DataFrame, ventana: int | None = None) -> pd.DataFrame:
    """Frecuencia de cada balota (1..43) en todo el histórico o en los últimos `ventana` sorteos."""
    d = df.tail(ventana) if ventana else df
    conteo = pd.Series(d[COLS_BALOTAS].values.ravel()).value_counts().reindex(range(1, N_BALOTAS + 1), fill_value=0)
    esperado = len(d) * K_ESCOGIDAS / N_BALOTAS
    ultimo_idx = {}
    for pos, (_, fila) in enumerate(d.iterrows()):
        for b in fila[COLS_BALOTAS]:
            ultimo_idx[int(b)] = pos
    rezago = pd.Series({b: len(d) - 1 - ultimo_idx.get(b, -1) for b in range(1, N_BALOTAS + 1)})
    out = pd.DataFrame({"Balota": conteo.index, "Frecuencia": conteo.values, "Esperada": round(esperado, 1),
                        "Desvio_%": ((conteo.values - esperado) / esperado * 100).round(1),
                        "SorteosSinSalir": rezago.values})
    return out.sort_values("Frecuencia", ascending=False).reset_index(drop=True)

def frecuencias_sb(df: pd.DataFrame, ventana: int | None = None) -> pd.DataFrame:
    d = df.tail(ventana) if ventana else df
    conteo = d["SB"].value_counts().reindex(range(1, N_SUPERBALOTA + 1), fill_value=0)
    return pd.DataFrame({"SB": conteo.index, "Frecuencia": conteo.values,
                         "Esperada": round(len(d) / N_SUPERBALOTA, 1)}).sort_values("Frecuencia", ascending=False).reset_index(drop=True)

def prueba_uniformidad(conteos, nombre: str) -> float:
    """χ² de bondad de ajuste contra la distribución uniforme. p alto → compatible con sorteo justo."""
    chi2, p = stats.chisquare(conteos)
    veredicto = "compatible con un sorteo uniforme (las diferencias son ruido)" if p > 0.05 else "⚠️ desviación estadísticamente significativa"
    print(f"   χ² {nombre}: estadístico={chi2:.1f}, p={p:.3f} → {veredicto}")
    return p

def resumen_descriptivo(df: pd.DataFrame, ventana: int = VENTANA_RECIENTE) -> dict:
    f_tot, f_rec = frecuencias(df), frecuencias(df, ventana)
    sb_tot, sb_rec = frecuencias_sb(df), frecuencias_sb(df, ventana)
    pares = (df[COLS_BALOTAS] % 2 == 0).sum(axis=1)
    sumas = df[COLS_BALOTAS].sum(axis=1)
    bajos = (df[COLS_BALOTAS] <= 21).sum(axis=1)   # mitad baja 1-21 / alta 22-43
    consecutivos = (np.diff(df[COLS_BALOTAS].values, axis=1) == 1).sum(axis=1)
    decenas = pd.Series(np.floor_divide(df[COLS_BALOTAS].values.ravel() - 1, 10)).value_counts().sort_index()
    r = {
        "n_sorteos": len(df),
        "calientes_recientes": f_rec.head(8)["Balota"].tolist(),
        "frios_recientes": f_rec.tail(8)["Balota"].tolist(),
        "mas_rezagados": f_tot.sort_values("SorteosSinSalir", ascending=False).head(6)[["Balota", "SorteosSinSalir"]].values.tolist(),
        "calientes_historicos": f_tot.head(8)["Balota"].tolist(),
        "frios_historicos": f_tot.tail(8)["Balota"].tolist(),
        "pares_dist": pares.value_counts().sort_index().to_dict(),
        "pares_media": pares.mean(),
        "suma_media": sumas.mean(), "suma_sd": sumas.std(), "suma_p10": sumas.quantile(.10), "suma_p90": sumas.quantile(.90),
        "bajos_media": bajos.mean(),
        "pct_con_consecutivos": (consecutivos > 0).mean() * 100,
        "sb_media": df["SB"].mean(), "sb_var": df["SB"].var(ddof=1), "sb_sd": df["SB"].std(ddof=1),
        "sb_var_teorica": (N_SUPERBALOTA ** 2 - 1) / 12, "sb_calientes": sb_rec.head(4)["SB"].tolist(), "sb_frios": sb_rec.tail(4)["SB"].tolist(),
        "decenas": decenas.to_dict(),
    }
    r["f_tot"], r["f_rec"], r["sb_tot"], r["sb_rec"] = f_tot, f_rec, sb_tot, sb_rec
    return r

df_an = _seleccionar(df_baloto, df_revancha, JUEGO_ANALISIS)
R = resumen_descriptivo(df_an)
exp_pares = {k: round(math.comb(21, k) * math.comb(22, 5 - k) / math.comb(43, 5) * 100, 1) for k in range(6)}  # 21 pares, 22 impares en 1..43

print(f"📊 RESUMEN ANALÍTICO — juego: {JUEGO_ANALISIS} | {R['n_sorteos']} sorteos | ventana reciente: {VENTANA_RECIENTE}\n")
print(f"🔥 Calientes (últimos {VENTANA_RECIENTE}): {R['calientes_recientes']}")
print(f"🧊 Fríos (últimos {VENTANA_RECIENTE}):     {R['frios_recientes']}")
print(f"⏳ Más sorteos sin salir:   {', '.join(f'{b} ({n})' for b, n in R['mas_rezagados'])}")
print(f"📈 Calientes histórico:     {R['calientes_historicos']}   | fríos histórico: {R['frios_historicos']}")
prueba_uniformidad(R["f_tot"]["Frecuencia"].values, "balotas (histórico completo)")
prueba_uniformidad(R["f_rec"]["Frecuencia"].values, f"balotas (últimos {VENTANA_RECIENTE})")
print(f"\n⚖️ Par/impar: media {R['pares_media']:.2f} pares de 5. Distribución observada % vs teórica %:")
for k in range(6):
    obs = R["pares_dist"].get(k, 0) / R["n_sorteos"] * 100
    print(f"   {k} pares / {5-k} impares: {obs:5.1f} %  (teórico {exp_pares[k]:5.1f} %)")
print(f"\n➕ Suma de las 5 balotas: media {R['suma_media']:.1f} ± {R['suma_sd']:.1f} | 80 % de los sorteos entre {R['suma_p10']:.0f} y {R['suma_p90']:.0f} (teórica 110)")
print(f"🔢 Balotas bajas (1-21) por sorteo: {R['bajos_media']:.2f} de 5 (teórico 2.44) | sorteos con al menos un par consecutivo: {R['pct_con_consecutivos']:.1f} % (teórico ≈ 40 %)")
print(f"\n🔴 Superbalota: media {R['sb_media']:.2f} (teórica 8.5) | varianza {R['sb_var']:.2f} (teórica uniforme {R['sb_var_teorica']:.2f}) | SD {R['sb_sd']:.2f}")
print(f"   SB calientes recientes: {R['sb_calientes']} | frías: {R['sb_frios']}")
prueba_uniformidad(R["sb_tot"]["Frecuencia"].values, "superbalota (histórico)")
print("\n📋 Tabla de frecuencias (top 10 histórico):")
print(R["f_tot"].head(10).to_string(index=False))


In [ ]:
# @title 5️⃣ Visualizaciones
import matplotlib.pyplot as plt
import matplotlib
if MODO_TEST_LOCAL:
    matplotlib.use("Agg")

plt.style.use("dark_background")
AZUL, AMARILLO, ROJO, GRIS = "#4cc9f0", "#ffd60a", "#ef476f", "#8d99ae"

def graficar_panel(R: dict, df: pd.DataFrame, guardar: Path | None = None):
    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"Baloto Master AI v6.0 — {R['n_sorteos']} sorteos ({JUEGO_ANALISIS})", fontsize=16, color="white")

    # 1. Frecuencia histórica por balota vs esperada
    f = R["f_tot"].sort_values("Balota")
    colores = [ROJO if b in R["calientes_recientes"] else (AZUL if b in R["frios_recientes"] else GRIS) for b in f["Balota"]]
    axs[0, 0].bar(f["Balota"], f["Frecuencia"], color=colores)
    axs[0, 0].axhline(f["Esperada"].iloc[0], color=AMARILLO, ls="--", label=f"esperada {f['Esperada'].iloc[0]}")
    axs[0, 0].set_title("Frecuencia histórica por balota (rojo = caliente reciente, azul = fría reciente)")
    axs[0, 0].set_xticks(range(1, 44, 3)); axs[0, 0].legend()

    # 2. Superbalota
    s = R["sb_tot"].sort_values("SB")
    axs[0, 1].bar(s["SB"], s["Frecuencia"], color=AMARILLO)
    axs[0, 1].axhline(s["Esperada"].iloc[0], color=ROJO, ls="--", label="esperada")
    axs[0, 1].set_title(f"Superbalota — varianza observada {R['sb_var']:.1f} vs teórica {R['sb_var_teorica']:.1f}")
    axs[0, 1].set_xticks(range(1, 17)); axs[0, 1].legend()

    # 3. Suma de las 5 balotas: observada vs teórica (por simulación)
    sumas = df[COLS_BALOTAS].sum(axis=1)
    rng = np.random.default_rng(0)
    sim = np.array([rng.choice(43, 5, replace=False).sum() + 5 for _ in range(50_000)])
    axs[1, 0].hist(sim, bins=range(15, 216, 5), density=True, alpha=.35, color=GRIS, label="teórica (simulada)")
    axs[1, 0].hist(sumas, bins=range(15, 216, 5), density=True, alpha=.7, color=AZUL, label="observada")
    axs[1, 0].axvline(R["suma_p10"], color=AMARILLO, ls=":"); axs[1, 0].axvline(R["suma_p90"], color=AMARILLO, ls=":", label="p10–p90")
    axs[1, 0].set_title("Suma de las 5 balotas"); axs[1, 0].legend()

    # 4. Pares por sorteo
    k = range(6)
    obs = [R["pares_dist"].get(i, 0) / R["n_sorteos"] * 100 for i in k]
    teo = [math.comb(21, i) * math.comb(22, 5 - i) / math.comb(43, 5) * 100 for i in k]
    axs[1, 1].bar([i - .2 for i in k], obs, width=.4, color=AZUL, label="observado %")
    axs[1, 1].bar([i + .2 for i in k], teo, width=.4, color=GRIS, label="teórico %")
    axs[1, 1].set_title("Cantidad de balotas pares por sorteo"); axs[1, 1].set_xticks(list(k)); axs[1, 1].legend()

    plt.tight_layout()
    if guardar:
        fig.savefig(guardar, dpi=110, bbox_inches="tight")
    if MODO_TEST_LOCAL:
        plt.close(fig)
    else:
        plt.show()
    return fig

graficar_panel(R, df_an, guardar=DATA_DIR / "panel_estadistico.png")
print("🖼️ Gráficos guardados en", DATA_DIR / "panel_estadistico.png")


In [ ]:
# @title 6️⃣ Probabilidades exactas, validación empírica y valor esperado (EV) del tiquete
# ─────────────────────────────────────────────────────────────────────────────
#  1. Probabilidad exacta de cada categoría (combinatoria).
#  2. Validación: ¿los ganadores reales por categoría coinciden con lo que predice
#     la probabilidad? (si sí, el sorteo es uniforme y ningún "patrón" lo cambia).
#  3. EV: cuánto vale en promedio un tiquete, con los premios oficiales recientes.
# ─────────────────────────────────────────────────────────────────────────────
TOTAL_COMB = math.comb(N_BALOTAS, K_ESCOGIDAS)

def prob_categoria(aciertos: int, con_sb: bool) -> float:
    p_b = math.comb(K_ESCOGIDAS, aciertos) * math.comb(N_BALOTAS - K_ESCOGIDAS, K_ESCOGIDAS - aciertos) / TOTAL_COMB
    return p_b * (1 / N_SUPERBALOTA if con_sb else (N_SUPERBALOTA - 1) / N_SUPERBALOTA)

# Cambio de reglas detectado en los datos: desde el sorteo #2192 (25-may-2022, nuevo operador / Acuerdo 01 de 2022)
# la categoría web "0 + SB" paga también con 1 balota + superbalota. Antes solo pagaba con 0 balotas + SB.
SORTEO_CAMBIO_REGLAS = 2192

def probabilidades_categorias(sorteo: int | None = None) -> dict:
    actual = sorteo is None or sorteo >= SORTEO_CAMBIO_REGLAS
    return {
        "5 + SB": prob_categoria(5, True), "5": prob_categoria(5, False),
        "4 + SB": prob_categoria(4, True), "4": prob_categoria(4, False),
        "3 + SB": prob_categoria(3, True), "3": prob_categoria(3, False),
        "2 + SB": prob_categoria(2, True),
        "0 + SB": prob_categoria(0, True) + (prob_categoria(1, True) if actual else 0.0),
    }

CATEGORIAS = probabilidades_categorias()          # reglas vigentes
tabla_prob = pd.DataFrame({"Categoria": CATEGORIAS.keys(), "Probabilidad": CATEGORIAS.values()})
tabla_prob["1 en"] = (1 / tabla_prob["Probabilidad"]).round(0).astype(int)
print("🎯 PROBABILIDADES EXACTAS (iguales para cualquier combinación):")
print(tabla_prob.assign(Probabilidad=lambda d: d.Probabilidad.map(lambda p: f"{p:.8%}")).to_string(index=False))
print(f"   Probabilidad de ganar algo: {sum(CATEGORIAS.values()):.2%}  (1 en {1/sum(CATEGORIAS.values()):.1f})")

# ── Validación empírica con los ganadores reales ────────────────────────────
def validar_uniformidad_con_ganadores(df_premios: pd.DataFrame, juego: str | None = None):
    d = df_premios if juego is None else df_premios[df_premios.Juego == juego]
    if d.empty:
        print("⚠️ Sin tabla de premios para validar."); return None
    # Excluimos el jackpot (muy raro) y comparamos proporciones entre categorías menores,
    # respetando el régimen de reglas de cada sorteo (antes/después de #2192).
    cats = [c for c in CATEGORIAS if c != "5 + SB"]
    o = d.groupby("Categoria")["Ganadores"].sum().reindex(cats, fill_value=0).values.astype(float)
    esperado = np.zeros(len(cats))
    for regimen, sub in d.groupby(d.Sorteo >= SORTEO_CAMBIO_REGLAS):
        pr = probabilidades_categorias(SORTEO_CAMBIO_REGLAS if regimen else SORTEO_CAMBIO_REGLAS - 1)
        p = np.array([pr[c] for c in cats]); p = p / p.sum()
        esperado += sub[sub.Categoria.isin(cats)]["Ganadores"].sum() * p
    tabla = pd.DataFrame({"Categoria": cats, "GanadoresObs": o.astype(int), "GanadoresEsperados": esperado.round(0).astype(int),
                          "Razon": (o / esperado).round(3)})
    # Tiquetes vendidos estimados por sorteo = ganadores / P(ganar algo) (régimen de cada sorteo)
    g = d.groupby(["Juego", "Sorteo"])["Ganadores"].sum()
    p_algo = g.index.get_level_values("Sorteo").map(lambda s: sum(probabilidades_categorias(int(s)).values()))
    tiquetes_est = g / np.asarray(p_algo)
    print(f"\n🧪 VALIDACIÓN EMPÍRICA ({juego or 'ambos juegos'}, {d.Sorteo.nunique()} sorteos): ganadores reales vs. esperados por probabilidad pura")
    print(tabla.to_string(index=False))
    desv = (tabla["Razon"] - 1).abs().max()
    print("   " + ("Razón ≈ 1.00 en todas las categorías ⇒ el sorteo se comporta como una lotería uniforme; ningún patrón cambia las probabilidades."
                   if desv < 0.15 else f"⚠️ Alguna categoría se desvía {desv:.0%} (normal con pocos sorteos; con cientos debe acercarse a 1.00)."))
    pm = d[(d.Categoria == "0 + SB") & (d.Ganadores > 0)].sort_values("Sorteo")["PremioPorGanador"]
    if len(pm):
        print(f"   💡 Premio mínimo oficial (0 + SB) en el último sorteo: {fmt_cop(pm.iloc[-1])}. Suele igualar el precio del tiquete: "
              f"si difiere de PRECIO_BALOTO ({fmt_cop(PRECIO_BALOTO)}), actualiza la constante en la celda 1.")
    print(f"   Tiquetes vendidos estimados por sorteo: mediana {tiquetes_est.median():,.0f} (rango {tiquetes_est.quantile(.1):,.0f}–{tiquetes_est.quantile(.9):,.0f})")
    return tabla, tiquetes_est

if not df_premios.empty:
    _val = validar_uniformidad_con_ganadores(df_premios)

# ── Valor esperado del tiquete ──────────────────────────────────────────────
def valor_esperado(df_premios: pd.DataFrame, juego: str, acumulado_millones: float, precio: float,
                   ultimos: int = 40, impuesto: float = IMPUESTO_GANANCIA_OCASIONAL) -> pd.DataFrame:
    d = df_premios[(df_premios.Juego == juego)]
    d = d[d.Sorteo >= d.Sorteo.max() - ultimos] if not d.empty else d
    filas = []
    for cat, p in CATEGORIAS.items():
        if cat == "5 + SB":
            premio = acumulado_millones * 1e6
        else:
            g = d[(d.Categoria == cat) & (d.Ganadores > 0)]["PremioPorGanador"]
            premio = float(g.median()) if len(g) else 0.0
        neto = premio * (1 - impuesto) if premio > 48 * 49_799 else premio   # UVT 2025 ≈ $49.799; umbral 48 UVT
        filas.append({"Categoria": cat, "Probabilidad": p, "PremioTipico": premio, "PremioNeto": neto, "Aporte_EV": p * neto})
    t = pd.DataFrame(filas)
    ev = t["Aporte_EV"].sum()
    print(f"\n💰 VALOR ESPERADO — {juego.upper()} | acumulado {fmt_cop(acumulado_millones*1e6)} | precio {fmt_cop(precio)}")
    print(t.assign(Probabilidad=lambda x: x.Probabilidad.map(lambda p: f"{p:.6%}"),
                   PremioTipico=lambda x: x.PremioTipico.map(fmt_cop), PremioNeto=lambda x: x.PremioNeto.map(fmt_cop),
                   Aporte_EV=lambda x: x.Aporte_EV.map(lambda v: f"${v:,.0f}")).to_string(index=False))
    print(f"   ➜ EV por tiquete: {fmt_cop(ev)}  → retorno {ev/precio*100:.1f} % del precio (pérdida esperada {fmt_cop(precio-ev)} por tiquete)")
    acum_be = (precio - (ev - t.loc[t.Categoria=='5 + SB','Aporte_EV'].iloc[0])) / CATEGORIAS["5 + SB"] / (1 - impuesto) / 1e6
    print(f"   ➜ El tiquete tendría EV ≥ precio solo con un acumulado ≥ {acum_be:,.0f} millones (sin contar que se comparte).")
    return t

if not df_premios.empty and not df_baloto.empty:
    ACUM_BALOTO = float(df_baloto.iloc[-1]["AcumuladoMillones"])
    ACUM_REVANCHA = float(df_revancha.iloc[-1]["AcumuladoMillones"]) if not df_revancha.empty else 0.0
    print("\nℹ️ El acumulado mostrado es el del ÚLTIMO sorteo descargado; si no cayó, el próximo es mayor (ver baloto.com).")
    ev_baloto = valor_esperado(df_premios, "baloto", ACUM_BALOTO, PRECIO_BALOTO)
    if ACUM_REVANCHA:
        ev_revancha = valor_esperado(df_premios, "revancha", ACUM_REVANCHA, PRECIO_CON_REVANCHA - PRECIO_BALOTO)


In [ ]:
# @title 7️⃣ Motor v6 — generación ponderada de tickets (único motor de la app)
# ─────────────────────────────────────────────────────────────────────────────
#  BalotoMasterEngineV6 reemplaza las 18 clases de la v5.0.
#
#  Pesos de cada balota = mezcla explícita de tres señales (ajustables):
#     • frecuencia reciente  (números "calientes")
#     • rezago               (sorteos sin salir, números "atrasados")
#     • uniforme             (la verdad matemática: todas valen 1/43)
#  Luego aplica FILTROS DE FORMA (suma típica, mezcla par/impar, no 5 bajas,
#  no 5 consecutivas, no repetir un ganador histórico) cuyo objetivo real es
#  evitar combinaciones POPULARES: si el premio llega, se comparte con menos gente.
#
#  Cada ticket se entrega con un ÍNDICE DE UNICIDAD (0-100) en lugar de la
#  "confianza 94 %" ficticia de la v5: mide cuán poco se parece a lo que la
#  mayoría de jugadores elige (fechas, secuencias, cumpleaños…).
# ─────────────────────────────────────────────────────────────────────────────

class BalotoMasterEngineV6:
    VERSION = "6.0.0"

    def __init__(self, df_hist: pd.DataFrame, ventana: int = 60,
                 peso_frecuencia: float = 0.4, peso_rezago: float = 0.2, peso_uniforme: float = 0.4,
                 excluir_ultimo_sorteo: bool = True, semilla: int | None = None):
        assert abs(peso_frecuencia + peso_rezago + peso_uniforme - 1) < 1e-9, "los pesos deben sumar 1"
        self.df = df_hist.sort_values(["Fecha", "Sorteo"]).reset_index(drop=True)
        self.ventana, self.pf, self.pr, self.pu = ventana, peso_frecuencia, peso_rezago, peso_uniforme
        self.excluir_ultimo = excluir_ultimo_sorteo
        self.rng = np.random.default_rng(semilla)
        self.ganadores_hist = {tuple(r) for r in self.df[COLS_BALOTAS].values.tolist()}
        self._calcular_pesos()

    # ── 1. pesos ─────────────────────────────────────────────────────────────
    def _calcular_pesos(self):
        d = self.df.tail(self.ventana)
        uniforme = np.full(N_BALOTAS, 1 / N_BALOTAS)
        if len(d) < 10:            # sin histórico suficiente → azar puro (nunca inventamos señal)
            self.w, self.w_sb = uniforme.copy(), np.full(N_SUPERBALOTA, 1 / N_SUPERBALOTA)
            self.freq_rec, self.rezago = np.zeros(N_BALOTAS), np.zeros(N_BALOTAS)
            self.suma_min, self.suma_max = 70, 150
            return
        vals = d[COLS_BALOTAS].values.astype(int)
        freq = np.bincount(vals.ravel(), minlength=N_BALOTAS + 1)[1:].astype(float)
        rez = np.zeros(N_BALOTAS)
        for b in range(1, N_BALOTAS + 1):
            apar = np.where((vals == b).any(axis=1))[0]
            rez[b - 1] = len(d) - 1 - apar[-1] if len(apar) else len(d)
        def norm(v):
            v = v + 1e-9
            return v / v.sum()
        self.w = self.pf * norm(freq) + self.pr * norm(rez) + self.pu * uniforme
        if self.excluir_ultimo:
            for b in d.iloc[-1][COLS_BALOTAS]:
                self.w[int(b) - 1] *= 0.5     # las balotas del sorteo anterior son de las más jugadas por el público → se atenúan
        self.w /= self.w.sum()
        sb_freq = np.bincount(d["SB"].values.astype(int), minlength=N_SUPERBALOTA + 1)[1:].astype(float)
        self.w_sb = self.pf * norm(sb_freq) + (self.pr + self.pu) * np.full(N_SUPERBALOTA, 1 / N_SUPERBALOTA)
        self.w_sb /= self.w_sb.sum()
        sumas = self.df[COLS_BALOTAS].sum(axis=1)
        self.suma_min, self.suma_max = sumas.quantile(.10), sumas.quantile(.90)
        self.freq_rec = freq; self.rezago = rez

    def tabla_pesos(self) -> pd.DataFrame:
        return pd.DataFrame({"Balota": range(1, N_BALOTAS + 1), "Peso_%": (self.w * 100).round(2),
                             "FrecReciente": self.freq_rec.astype(int), "SorteosSinSalir": self.rezago.astype(int)}
                            ).sort_values("Peso_%", ascending=False).reset_index(drop=True)

    # ── 2. filtros de forma / popularidad ───────────────────────────────────
    def pasa_filtros(self, b: list[int]) -> bool:
        b = sorted(b)
        pares = sum(x % 2 == 0 for x in b)
        if not (1 <= pares <= 4): return False                         # evita 5 pares / 5 impares (3 % de los sorteos)
        if not (self.suma_min <= sum(b) <= self.suma_max): return False   # suma dentro del 80 % central
        if all(x <= 31 for x in b): return False                       # "todas fechas": muy popular
        if sum(np.diff(b) == 1) >= 2: return False                     # evita 3+ consecutivas
        if tuple(b) in self.ganadores_hist: return False               # nunca repetir un ganador histórico
        if len({x // 10 for x in b}) < 3: return False                 # al menos 3 decenas distintas
        return True

    @staticmethod
    def indice_unicidad(b: list[int], sb: int) -> float:
        """0-100. Penaliza rasgos que la gente elige mucho (fechas ≤31, ≤12, consecutivas, múltiplos, SB 7)."""
        b = sorted(b); score = 100.0
        score -= 6 * sum(x <= 31 for x in b)             # días de mes
        score -= 4 * sum(x <= 12 for x in b)             # meses
        score -= 10 * int(sum(np.diff(b) == 1))          # consecutivas
        score -= 8 if len({x % 10 for x in b}) <= 2 else 0   # mismas terminaciones
        score -= 8 if all(x % 5 == 0 for x in b) else 0
        score -= 5 if sb in (7, 3, 13) else 0            # SB "de la suerte"
        score += 4 * sum(x >= 32 for x in b)             # números altos son menos elegidos
        return float(np.clip(score, 0, 100))

    # ── 3. generación ────────────────────────────────────────────────────────
    def generar_ticket(self, temperatura: float = 1.0, filtros: bool = True, max_intentos: int = 1000) -> dict:
        w = self.w ** (1 / temperatura); w /= w.sum()
        for _ in range(max_intentos):
            b = sorted(self.rng.choice(np.arange(1, N_BALOTAS + 1), K_ESCOGIDAS, replace=False, p=w).tolist())
            if not filtros or self.pasa_filtros(b):
                break
        sb = int(self.rng.choice(np.arange(1, N_SUPERBALOTA + 1), p=self.w_sb))
        return self.describir(b, sb)

    def describir(self, b: list[int], sb: int) -> dict:
        b = sorted(int(x) for x in b)
        return {"balotas": b, "sb": int(sb), "suma": sum(b), "pares": sum(x % 2 == 0 for x in b),
                "calientes": sum(1 for x in b if self.freq_rec[x - 1] >= np.percentile(self.freq_rec, 75)),
                "rezagadas": sum(1 for x in b if self.rezago[x - 1] >= np.percentile(self.rezago, 75)),
                "unicidad": round(self.indice_unicidad(b, sb), 1)}

    def generar_tickets(self, n: int = 4, temperatura_base: float = 1.0, paso_temperatura: float = 0.25,
                        filtros: bool = True, max_solapamiento: int = 2) -> list[dict]:
        """n tickets distintos; los primeros siguen más los pesos, los últimos se acercan al azar puro."""
        tickets = []
        intentos = 0
        while len(tickets) < n and intentos < 500:
            intentos += 1
            t = self.generar_ticket(temperatura_base + paso_temperatura * len(tickets), filtros)
            if any(len(set(t["balotas"]) & set(u["balotas"])) > max_solapamiento for u in tickets):
                continue        # tickets poco solapados → cubren más números distintos
            t["id"] = len(tickets) + 1
            tickets.append(t)
        return tickets

    def evaluar(self, b: list[int], sb: int, resultado_b: list[int], resultado_sb: int, sorteo: int | None = None) -> str:
        """Categoría de premio según las reglas vigentes (desde #2192 "0 + SB" incluye 1 balota + SB)."""
        ac = len(set(b) & set(resultado_b)); con_sb = int(sb) == int(resultado_sb)
        if ac == 5: return "5 + SB" if con_sb else "5"
        if ac == 4: return "4 + SB" if con_sb else "4"
        if ac == 3: return "3 + SB" if con_sb else "3"
        if ac == 2 and con_sb: return "2 + SB"
        reglas_actuales = sorteo is None or sorteo >= SORTEO_CAMBIO_REGLAS
        if con_sb and (ac == 0 or (ac == 1 and reglas_actuales)): return "0 + SB"
        return "—"


def tabla_tickets(tickets: list[dict]) -> pd.DataFrame:
    return pd.DataFrame([{"Ticket": t["id"], "Balotas": " - ".join(f"{x:02d}" for x in t["balotas"]), "SB": f"{t['sb']:02d}",
                          "Suma": t["suma"], "Pares": f"{t['pares']}/5", "Calientes": t["calientes"], "Rezagadas": t["rezagadas"],
                          "Unicidad": f"{t['unicidad']:.0f}/100"} for t in tickets])

# ── Instanciar y generar ────────────────────────────────────────────────────
PESOS = dict(peso_frecuencia=0.4, peso_rezago=0.2, peso_uniforme=0.4)   # ← ajusta aquí la "personalidad" del motor
NUM_TICKETS = 4
motor = BalotoMasterEngineV6(df_an, ventana=60, **PESOS)
tickets = motor.generar_tickets(NUM_TICKETS)
_ps = proximo_sorteo()

print(f"🎟️ TABLA 2 — {NUM_TICKETS} tickets para el sorteo del {DIAS_SORTEO[_ps.weekday()]} {_ps:%d-%m-%Y} (Baloto y Revancha usan los mismos números)")
print(tabla_tickets(tickets).to_string(index=False))
print("\n⚠️ Recordatorio honesto: cada ticket tiene exactamente 1/15.401.568 de ganar el mayor, igual que cualquier otro.")
print("   La ponderación y los filtros buscan combinaciones MENOS POPULARES (menos probabilidad de compartir premio), no 'más probables'.")
print(f"\n🔍 Top 10 pesos del motor (ventana {motor.ventana}):")
print(motor.tabla_pesos().head(10).to_string(index=False))


In [ ]:
# @title 8️⃣ Backtesting walk-forward: motor v6 vs. azar puro (Quick Pick)
# ─────────────────────────────────────────────────────────────────────────────
#  Para cada uno de los últimos N sorteos: entrena el motor SOLO con los sorteos
#  anteriores, genera 4 tickets, y cuenta aciertos contra el resultado real.
#  Lo compara con 4 tickets totalmente aleatorios. Si el motor "funcionara",
#  debería ganar de forma consistente. Spoiler: la diferencia no será significativa,
#  y eso es exactamente lo que la teoría predice. Este módulo existe para que la
#  app nunca vuelva a prometer lo que no puede cumplir.
# ─────────────────────────────────────────────────────────────────────────────
N_BACKTEST = 150
TICKETS_POR_SORTEO = 4

def backtest(df: pd.DataFrame, n_sorteos: int = N_BACKTEST, k: int = TICKETS_POR_SORTEO, semilla: int = 42, **pesos) -> pd.DataFrame:
    df = df.sort_values(["Fecha", "Sorteo"]).reset_index(drop=True)
    rng = np.random.default_rng(semilla)
    filas = []
    n_sorteos = min(n_sorteos, max(len(df) - 30, 1))     # deja al menos 30 sorteos de "entrenamiento" inicial
    for i in range(len(df) - n_sorteos, len(df)):
        hist, real = df.iloc[:i], df.iloc[i]
        m = BalotoMasterEngineV6(hist, semilla=int(rng.integers(1e9)), **pesos)
        rb, rsb = [int(x) for x in real[COLS_BALOTAS]], int(real["SB"])
        for t in m.generar_tickets(k):
            ac = len(set(t["balotas"]) & set(rb))
            filas.append({"Sorteo": real.Sorteo, "Estrategia": "Motor v6", "Aciertos": ac, "SB": int(t["sb"] == rsb),
                          "Categoria": m.evaluar(t["balotas"], t["sb"], rb, rsb, sorteo=int(real.Sorteo))})
        for _ in range(k):
            b = sorted(rng.choice(np.arange(1, N_BALOTAS + 1), K_ESCOGIDAS, replace=False).tolist()); sb = int(rng.integers(1, N_SUPERBALOTA + 1))
            filas.append({"Sorteo": real.Sorteo, "Estrategia": "Azar (Quick Pick)", "Aciertos": len(set(b) & set(rb)),
                          "SB": int(sb == rsb), "Categoria": m.evaluar(b, sb, rb, rsb, sorteo=int(real.Sorteo))})
    return pd.DataFrame(filas)

bt = backtest(df_an, **PESOS)
res = bt.groupby("Estrategia").agg(Tickets=("Aciertos", "size"), AciertosMedios=("Aciertos", "mean"),
                                   Tickets_3mas=("Aciertos", lambda s: (s >= 3).sum()), SB_aciertos=("SB", "sum"),
                                   ConPremio=("Categoria", lambda s: (s != "—").sum()))
res["Esperado_AciertosMedios"] = K_ESCOGIDAS * K_ESCOGIDAS / N_BALOTAS
res["Esperado_ConPremio"] = (res["Tickets"] * sum(CATEGORIAS.values())).round(1)
print(f"🔁 BACKTEST — últimos {bt.Sorteo.nunique()} sorteos × {TICKETS_POR_SORTEO} tickets por estrategia\n")
print(res.round(3).to_string())

a, b_ = bt[bt.Estrategia == "Motor v6"]["Aciertos"], bt[bt.Estrategia != "Motor v6"]["Aciertos"]
_, p_mw = stats.mannwhitneyu(a, b_, alternative="two-sided")
ct = pd.crosstab(bt.Estrategia, bt.Categoria != "—")
_, p_chi, *_ = stats.chi2_contingency(ct) if ct.shape == (2, 2) else (None, 1.0)
print(f"\n📐 Aciertos medios: motor {a.mean():.3f} vs azar {b_.mean():.3f} (teórico {25/43:.3f}) → Mann-Whitney p={p_mw:.3f}")
print(f"📐 Tickets con premio: motor {int((bt[bt.Estrategia=='Motor v6'].Categoria!='—').sum())} vs azar "
      f"{int((bt[bt.Estrategia!='Motor v6'].Categoria!='—').sum())} → χ² p={p_chi:.3f}")
veredicto = ("No hay diferencia estadísticamente significativa: el motor no predice mejor que el azar (como dicta la teoría). "
             "Su valor está en evitar combinaciones populares y en la disciplina de juego, no en 'acertar más'."
             if min(p_mw, p_chi) > 0.05 else
             "⚠️ Diferencia significativa en esta muestra. Antes de celebrar: repite con otra semilla y otra ventana; "
             "con muchas pruebas, 1 de cada 20 saldrá 'significativa' por puro azar.")
print(f"\n🧾 Veredicto: {veredicto}")
print("\n📊 Categorías obtenidas en el backtest:")
print(pd.crosstab(bt.Estrategia, bt.Categoria).to_string())


In [ ]:
# @title 9️⃣ OCR — leer imágenes de resultados o de tiquetes jugados (opcional)
# ─────────────────────────────────────────────────────────────────────────────
#  • Sube una captura de resultados (p. ej. de baloto.com o prensa) → Tabla 1 con
#    fecha, balotas y superbalota, lista para revisar y, si quieres, agregar al histórico.
#  • Sube la foto/PDF-captura de tu tiquete → extrae tus combinaciones para registrarlas.
#  REGLA v6: si el OCR no está seguro, lo dice. NUNCA inventa números (la v5 rellenaba al azar).
# ─────────────────────────────────────────────────────────────────────────────
try:
    import cv2, pytesseract
    OCR_DISPONIBLE = shutil.which("tesseract") is not None
except ImportError:
    OCR_DISPONIBLE = False
if not OCR_DISPONIBLE:
    print("ℹ️ OCR no disponible en este entorno (falta cv2/pytesseract/tesseract). Las funciones quedan definidas pero inactivas.")

_RE_FECHA = re.compile(r"(\d{1,2})[/\-. ](\d{1,2}|[A-Za-záé]{3,10})[/\-. ](\d{2,4})")

def preprocesar_imagen(ruta: str, escala: float = 2.0):
    img = cv2.imread(str(ruta))
    if img is None:
        raise FileNotFoundError(ruta)
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gris = cv2.resize(gris, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)
    gris = cv2.bilateralFilter(gris, 9, 75, 75)
    # Dos binarizaciones: normal e invertida (las balotas suelen ser texto claro sobre bola oscura)
    th = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10)
    return [th, cv2.bitwise_not(th)]

def ocr_texto(ruta: str) -> str:
    if not OCR_DISPONIBLE:
        return ""
    textos = []
    for im in preprocesar_imagen(ruta):
        for psm in (6, 4, 11):
            textos.append(pytesseract.image_to_string(im, lang="spa+eng", config=f"--psm {psm}"))
    return "\n".join(textos)

def _parsear_fecha(s: str):
    m = _RE_FECHA.search(s)
    if not m:
        return None
    d, mth, y = m.groups()
    try:
        mes = int(mth) if mth.isdigit() else MESES.get(mth.lower()[:3] if mth.lower()[:3] not in ("mar",) else "marzo", None)
        if mes is None:
            mes = next((v for k, v in MESES.items() if k.startswith(mth.lower()[:3])), None)
        y = int(y) if len(y) == 4 else 2000 + int(y)
        return date(y, int(mes), int(d))
    except (ValueError, TypeError):
        return None

def extraer_combinaciones(texto: str) -> list[dict]:
    """Busca líneas con 6 números (5 balotas 1-43 ordenadas + SB 1-16). Devuelve candidatos con nivel de confianza."""
    out, fecha_ctx = [], None
    for linea in texto.splitlines():
        f = _parsear_fecha(linea)
        if f:
            fecha_ctx = f
        nums = [int(n) for n in re.findall(r"(?<!\d)(\d{1,2})(?!\d)", linea)]
        for i in range(len(nums) - 5):
            b, sb = nums[i:i + 5], nums[i + 5]
            if validar_combinacion(b, sb):
                conf = "alta" if b == sorted(b) else "media (balotas no ordenadas)"
                out.append({"Fecha": fecha_ctx, "Balotas": sorted(b), "SB": sb, "Confianza": conf, "Linea": linea.strip()[:80]})
    # dedupe
    vistos, res = set(), []
    for o in out:
        k = (tuple(o["Balotas"]), o["SB"])
        if k not in vistos:
            vistos.add(k); res.append(o)
    return res

def tabla_1_desde_imagenes(rutas: list[str]) -> pd.DataFrame:
    filas = []
    for r in rutas:
        try:
            for c in extraer_combinaciones(ocr_texto(r)):
                filas.append({"Archivo": Path(r).name, **c})
        except Exception as e:
            print(f"⚠️ {r}: {e}")
    df = pd.DataFrame(filas)
    if df.empty:
        print("❌ No se detectaron combinaciones válidas. Revisa la imagen (recorta la zona de los números, más contraste) y vuelve a intentar.")
    else:
        print("📋 TABLA 1 — Datos extraídos por OCR (REVISA cada fila antes de usarla):")
        print(df.assign(Balotas=lambda d: d.Balotas.map(lambda b: " - ".join(f"{x:02d}" for x in b))).to_string(index=False))
    return df

def agregar_resultados_ocr_al_historico(df_ocr: pd.DataFrame, juego: str, sorteos: list[int]):
    """Agrega filas OCR al histórico oficial SOLO con fecha válida, sorteo indicado y sin duplicar."""
    assert len(sorteos) == len(df_ocr), "indica un número de sorteo por fila"
    df = _cargar_csv(ARCHIVOS[juego]); nuevas = []
    for (_, r), s in zip(df_ocr.iterrows(), sorteos):
        if r.Fecha is None or (not df.empty and s in set(df.Sorteo)):
            print(f"⏭️ Sorteo {s}: omitido (sin fecha o ya existe)."); continue
        nuevas.append({"Juego": juego, "Sorteo": s, "Fecha": r.Fecha, "DiaSemana": DIAS_SORTEO.get(pd.Timestamp(r.Fecha).weekday(), ""),
                       **dict(zip(COLS_BALOTAS, r.Balotas)), "SB": r.SB, "AcumuladoMillones": np.nan, "TotalGanadores": np.nan, "TotalPremiacion": np.nan})
    if nuevas:
        df = pd.concat([df, pd.DataFrame(nuevas)], ignore_index=True).sort_values("Sorteo")
        df.to_csv(ARCHIVOS[juego], index=False)
        print(f"✅ {len(nuevas)} sorteos agregados a {ARCHIVOS[juego].name}. Vuelve a correr la celda 3 para recargar.")

def subir_y_leer_imagenes() -> pd.DataFrame:
    """En Colab abre el selector de archivos; devuelve la Tabla 1."""
    if not EN_COLAB:
        print("Fuera de Colab usa: tabla_1_desde_imagenes(['ruta/imagen.png'])"); return pd.DataFrame()
    from google.colab import files
    subidos = files.upload()
    return tabla_1_desde_imagenes(list(subidos.keys()))

if OCR_DISPONIBLE:
    print("✅ OCR listo. Ejecuta:  df_ocr = subir_y_leer_imagenes()")
    print("   Para tiquetes jugados: luego  registrar_jugadas_desde_ocr(df_ocr, sorteo=NNNN)  (celda 10)")


In [ ]:
# @title 🔟 Mis jugadas — registro, verificación automática y balance personal
# ─────────────────────────────────────────────────────────────────────────────
#  Guarda cada tiquete jugado (manual, desde el motor o desde OCR) en Drive,
#  lo cruza con los resultados oficiales de Baloto y Revancha y calcula el premio
#  por categoría usando la tabla real de premios de ese sorteo. Balance = premios − costo.
# ─────────────────────────────────────────────────────────────────────────────
COLS_JUGADAS = ["FechaRegistro", "Sorteo", "B1", "B2", "B3", "B4", "B5", "SB", "ConRevancha", "Costo", "Origen"]

def cargar_jugadas() -> pd.DataFrame:
    df = _cargar_csv(ARCHIVOS["jugadas"])
    return df if not df.empty else pd.DataFrame(columns=COLS_JUGADAS)

def registrar_jugada(balotas, sb, sorteo: int, con_revancha: bool = True, costo: float | None = None, origen: str = "manual") -> pd.DataFrame:
    if not validar_combinacion(balotas, sb):
        raise ValueError(f"Combinación inválida: {balotas} + {sb}")
    costo = costo if costo is not None else (PRECIO_CON_REVANCHA if con_revancha else PRECIO_BALOTO)
    df = cargar_jugadas()
    fila = {"FechaRegistro": datetime.now().strftime("%Y-%m-%d %H:%M"), "Sorteo": int(sorteo), **dict(zip(COLS_BALOTAS, sorted(int(x) for x in balotas))),
            "SB": int(sb), "ConRevancha": bool(con_revancha), "Costo": float(costo), "Origen": origen}
    df = pd.concat([df, pd.DataFrame([fila])], ignore_index=True)
    df.to_csv(ARCHIVOS["jugadas"], index=False)
    print(f"💾 Jugada registrada para el sorteo #{sorteo}: {fila['B1']}-{fila['B2']}-{fila['B3']}-{fila['B4']}-{fila['B5']} + SB {sb} ({'con' if con_revancha else 'sin'} Revancha, {fmt_cop(costo)})")
    return df

def registrar_tickets_del_motor(tickets: list[dict], sorteo: int, con_revancha: bool = True):
    for t in tickets:
        registrar_jugada(t["balotas"], t["sb"], sorteo, con_revancha, origen=f"motor v{BalotoMasterEngineV6.VERSION}")

def registrar_jugadas_desde_ocr(df_ocr: pd.DataFrame, sorteo: int, con_revancha: bool = True):
    for _, r in df_ocr.iterrows():
        registrar_jugada(r.Balotas, r.SB, sorteo, con_revancha, origen=f"OCR:{r.get('Archivo','')}")

def _premio_categoria(df_premios: pd.DataFrame, juego: str, sorteo: int, cat: str, acumulado_millones: float = np.nan) -> float:
    d = df_premios[(df_premios.Juego == juego) & (df_premios.Sorteo == sorteo) & (df_premios.Categoria == cat)]
    premio = float(d["PremioPorGanador"].iloc[0]) if len(d) else np.nan
    if cat == "5 + SB" and (not premio or np.isnan(premio)) and not np.isnan(acumulado_millones):
        premio = acumulado_millones * 1e6      # nadie ganó ese día: el premio habría sido el acumulado completo
    return premio

def verificar_jugadas(df_b: pd.DataFrame = None, df_r: pd.DataFrame = None, df_p: pd.DataFrame = None) -> pd.DataFrame:
    df_b = df_baloto if df_b is None else df_b; df_r = df_revancha if df_r is None else df_r; df_p = df_premios if df_p is None else df_p
    j = cargar_jugadas()
    if j.empty:
        print("ℹ️ No hay jugadas registradas. Usa registrar_jugada([..5 balotas..], sb, sorteo)."); return j
    filas = []
    for _, r in j.iterrows():
        b, sb, s = [int(r[c]) for c in COLS_BALOTAS], int(r.SB), int(r.Sorteo)
        fila = {**r.to_dict(), "Estado": "pendiente", "CatBaloto": "—", "PremioBaloto": 0.0, "CatRevancha": "—", "PremioRevancha": 0.0}
        rb = df_b[df_b.Sorteo == s]
        if len(rb):
            rb = rb.iloc[0]; cat = motor.evaluar(b, sb, [int(x) for x in rb[COLS_BALOTAS]], int(rb.SB), sorteo=s)
            fila.update(Estado="sorteado", CatBaloto=cat,
                        PremioBaloto=_premio_categoria(df_p, "baloto", s, cat, float(rb.AcumuladoMillones)) if cat != "—" else 0.0)
            if r.ConRevancha:
                rr = df_r[df_r.Sorteo == s]
                if len(rr):
                    rr = rr.iloc[0]; cat_r = motor.evaluar(b, sb, [int(x) for x in rr[COLS_BALOTAS]], int(rr.SB), sorteo=s)
                    fila.update(CatRevancha=cat_r,
                                PremioRevancha=_premio_categoria(df_p, "revancha", s, cat_r, float(rr.AcumuladoMillones)) if cat_r != "—" else 0.0)
        filas.append(fila)
    out = pd.DataFrame(filas)
    out["PremioTotal"] = out[["PremioBaloto", "PremioRevancha"]].fillna(0).sum(axis=1)
    sort = out[out.Estado == "sorteado"]
    print(f"🎟️ MIS JUGADAS: {len(out)} tiquetes | {len(sort)} ya sorteados | {len(out)-len(sort)} pendientes")
    if len(sort):
        inv, prem = sort.Costo.sum(), sort.PremioTotal.sum()
        print(f"💵 Invertido: {fmt_cop(inv)} | Premios: {fmt_cop(prem)} | Balance: {fmt_cop(prem-inv)} ({(prem/inv-1)*100:+.1f} %)")
    cols = ["Sorteo", *COLS_BALOTAS, "SB", "Estado", "CatBaloto", "PremioBaloto", "CatRevancha", "PremioRevancha", "Costo", "Origen"]
    print(out[cols].tail(15).to_string(index=False))
    return out

print("✅ Módulo de jugadas listo.")
print("   registrar_jugada([3,17,22,35,41], 9, sorteo=2735)        → guarda un tiquete")
print(f"   registrar_tickets_del_motor(tickets, sorteo=NNNN)         → guarda los {len(tickets)} tickets de la celda 7")
print("   verificar_jugadas()                                      → cruza con resultados oficiales y calcula balance")


In [ ]:
# @title 1️⃣1️⃣ Panel interactivo (botón → tickets nuevos con métricas reales)
from IPython.display import HTML, display

def _estado_json() -> dict:
    ps = proximo_sorteo()
    ult_b = df_baloto.iloc[-1] if not df_baloto.empty else None
    ult_r = df_revancha.iloc[-1] if not df_revancha.empty else None
    return {
        "proximo": f"{DIAS_SORTEO[ps.weekday()]} {ps:%d-%m-%Y} · 11:00 p.m.",
        "ultimo_baloto": (f"#{ult_b.Sorteo} {ult_b.Fecha:%d-%m-%Y}: " + " · ".join(f"{int(x):02d}" for x in ult_b[COLS_BALOTAS]) + f" + {int(ult_b.SB):02d}") if ult_b is not None else "—",
        "ultimo_revancha": (" · ".join(f"{int(x):02d}" for x in ult_r[COLS_BALOTAS]) + f" + {int(ult_r.SB):02d}") if ult_r is not None else "—",
        "acum_baloto": f"{ult_b.AcumuladoMillones:,.0f}" if ult_b is not None else "—",
        "acum_revancha": f"{ult_r.AcumuladoMillones:,.0f}" if ult_r is not None else "—",
        "n_sorteos": int(len(df_hist)),
        "ev_txt": (f"EV ≈ {fmt_cop(ev_baloto.Aporte_EV.sum())} por tiquete de {fmt_cop(PRECIO_BALOTO)}" if "ev_baloto" in globals() else ""),
    }

def generar_tickets_panel(n: int = 4, pf: float = 0.4, pr: float = 0.2) -> dict:
    """Callback del panel. Devuelve un dict JSON-serializable (Colab lo expone como application/json)."""
    try:
        n = int(n); pf, pr = float(pf), float(pr); pu = max(0.0, 1 - pf - pr)
        m = BalotoMasterEngineV6(df_an, ventana=60, peso_frecuencia=pf, peso_rezago=pr, peso_uniforme=pu)
        return json.loads(json.dumps({"tickets": m.generar_tickets(n), "estado": _estado_json()}, default=str))
    except Exception as e:
        return {"error": str(e)}

if EN_COLAB:
    from google.colab import output
    output.register_callback("bm6.generar", generar_tickets_panel)

PANEL_HTML = """
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap" rel="stylesheet">
<style>
 .bm6{font-family:Inter,system-ui,sans-serif;max-width:960px;margin:0 auto;background:#0b1020;color:#e6ebff;border:1px solid #27324d;border-radius:18px;padding:26px}
 .bm6 h1{margin:0;font-size:26px;letter-spacing:.5px} .bm6 .v{background:#ffd60a;color:#0b1020;border-radius:999px;padding:2px 10px;font-size:13px;font-weight:800;margin-left:8px}
 .bm6 .grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px;margin:18px 0}
 .bm6 .kpi{background:#121a33;border:1px solid #27324d;border-radius:12px;padding:12px 14px} .bm6 .kpi small{color:#8d99ae;display:block;font-size:11px;text-transform:uppercase;letter-spacing:1px}
 .bm6 .kpi b{font-size:15px}
 .bm6 .ctrl{display:flex;gap:14px;flex-wrap:wrap;align-items:center;background:#121a33;border-radius:12px;padding:12px 14px;margin-bottom:14px}
 .bm6 label{font-size:12px;color:#8d99ae} .bm6 input{width:70px;background:#0b1020;color:#fff;border:1px solid #27324d;border-radius:8px;padding:5px}
 .bm6 button{background:linear-gradient(90deg,#4cc9f0,#4361ee);border:0;color:#fff;font-weight:800;padding:12px 22px;border-radius:12px;cursor:pointer;font-size:15px}
 .bm6 button:hover{filter:brightness(1.1)}
 .bm6 .tix{display:grid;grid-template-columns:repeat(auto-fit,minmax(290px,1fr));gap:14px}
 .bm6 .t{background:#121a33;border:1px solid #27324d;border-radius:14px;padding:14px} .bm6 .t .h{display:flex;justify-content:space-between;color:#8d99ae;font-size:12px;margin-bottom:10px}
 .bm6 .balls{display:flex;gap:8px;align-items:center;justify-content:center} .bm6 .b{width:40px;height:40px;border-radius:50%;background:#ffd60a;color:#0b1020;font-weight:800;display:flex;align-items:center;justify-content:center;font-size:17px}
 .bm6 .b.sb{background:#ef476f;color:#fff} .bm6 .meta{display:flex;justify-content:space-between;font-size:12px;color:#8d99ae;margin-top:10px}
 .bm6 .bar{height:6px;background:#27324d;border-radius:4px;margin-top:6px} .bm6 .bar i{display:block;height:6px;border-radius:4px;background:#4cc9f0}
 .bm6 .note{font-size:12px;color:#8d99ae;margin-top:14px;line-height:1.5}
</style>
<div class="bm6">
 <h1>🎱 Baloto Master AI <span class="v">v6.0</span></h1>
 <div class="grid" id="bm6-kpis"><div class="kpi"><small>Estado</small><b>Pulsa “Generar” para cargar</b></div></div>
 <div class="ctrl">
   <label>Tickets <input id="bm6-n" type="number" value="4" min="1" max="10"></label>
   <label>Peso frecuencia <input id="bm6-pf" type="number" value="0.4" step="0.1" min="0" max="1"></label>
   <label>Peso rezago <input id="bm6-pr" type="number" value="0.2" step="0.1" min="0" max="1"></label>
   <button onclick="bm6gen()">⚡ Generar tickets</button>
 </div>
 <div class="tix" id="bm6-tix"></div>
 <div class="note">Índice de unicidad = qué tan poco se parece el ticket a lo que la mayoría juega (fechas, secuencias). Todas las combinaciones tienen la misma probabilidad (1 en 15.401.568); el panel optimiza no compartir el premio, no “acertar más”. Juega con moderación.</div>
</div>
<script>
async function bm6gen(){
  const tix=document.getElementById('bm6-tix'); tix.innerHTML='<div class="t">⏳ Calculando…</div>';
  try{
    const r=await google.colab.kernel.invokeFunction('bm6.generar',[document.getElementById('bm6-n').value,document.getElementById('bm6-pf').value,document.getElementById('bm6-pr').value],{});
    let d=r.data['application/json'];
    if(!d){ let raw=r.data['text/plain']||''; raw=raw.replace(/^'|'$/g,'').replace(/\\\\'/g,"'"); d=JSON.parse(raw); }
    if(d.error){tix.innerHTML='<div class="t">❌ '+d.error+'</div>';return;}
    const e=d.estado;
    document.getElementById('bm6-kpis').innerHTML=
      `<div class="kpi"><small>Próximo sorteo</small><b>${e.proximo}</b></div>
       <div class="kpi"><small>Acumulado Baloto / Revancha</small><b>$${e.acum_baloto} M / $${e.acum_revancha} M</b></div>
       <div class="kpi"><small>Último Baloto</small><b>${e.ultimo_baloto}</b></div>
       <div class="kpi"><small>Última Revancha</small><b>${e.ultimo_revancha}</b></div>
       <div class="kpi"><small>Sorteos analizados</small><b>${e.n_sorteos}</b></div>
       <div class="kpi"><small>Valor esperado</small><b>${e.ev_txt||'—'}</b></div>`;
    tix.innerHTML=d.tickets.map(t=>`<div class="t"><div class="h"><span>TICKET ${t.id}</span><span>unicidad ${t.unicidad}/100</span></div>
      <div class="balls">${t.balotas.map(b=>`<div class="b">${String(b).padStart(2,'0')}</div>`).join('')}<span style="color:#8d99ae">+</span><div class="b sb">${String(t.sb).padStart(2,'0')}</div></div>
      <div class="bar"><i style="width:${t.unicidad}%"></i></div>
      <div class="meta"><span>suma ${t.suma}</span><span>${t.pares} pares</span><span>${t.calientes} calientes</span><span>${t.rezagadas} rezagadas</span></div></div>`).join('');
  }catch(err){tix.innerHTML='<div class="t">❌ Sin conexión con el kernel. Ejecuta primero las celdas 1 → 7.</div>';}
}
</script>"""

if EN_COLAB:
    display(HTML(PANEL_HTML))
else:
    print("ℹ️ El panel interactivo requiere Google Colab. Prueba local del callback:")
    _r = generar_tickets_panel(2); assert "tickets" in _r, _r
    print(_r["estado"]["proximo"], "|", [t["balotas"] for t in _r["tickets"]])


In [ ]:
# @title 1️⃣2️⃣ Exportar informe (Excel + PDF) y descargar
def exportar_informe(nombre: str | None = None) -> list[Path]:
    ps = proximo_sorteo()
    nombre = nombre or f"BalotoMasterAI_v6_{datetime.now():%Y%m%d_%H%M}"
    salidas = []
    # ── Excel ──
    xlsx = DATA_DIR / f"{nombre}.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
        tabla_tickets(tickets).to_excel(w, sheet_name="Tickets", index=False)
        R["f_tot"].to_excel(w, sheet_name="Frecuencias", index=False)
        R["f_rec"].to_excel(w, sheet_name=f"Frec_ult{VENTANA_RECIENTE}", index=False)
        R["sb_tot"].to_excel(w, sheet_name="Superbalota", index=False)
        tabla_prob.to_excel(w, sheet_name="Probabilidades", index=False)
        if "ev_baloto" in globals(): ev_baloto.to_excel(w, sheet_name="EV_Baloto", index=False)
        if "bt" in globals(): res.to_excel(w, sheet_name="Backtest")
        df_baloto.to_excel(w, sheet_name="Hist_Baloto", index=False); df_revancha.to_excel(w, sheet_name="Hist_Revancha", index=False)
        if not df_premios.empty: df_premios.to_excel(w, sheet_name="Premios", index=False)
        j = cargar_jugadas()
        if not j.empty: j.to_excel(w, sheet_name="MisJugadas", index=False)
    salidas.append(xlsx)
    # ── PDF (si fpdf2 está disponible) ──
    try:
        from fpdf import FPDF
        pdf = FPDF(); pdf.set_auto_page_break(auto=True, margin=15); pdf.add_page()
        pdf.set_font("Helvetica", "B", 16); pdf.cell(0, 10, "Baloto Master AI v6.0 - Informe", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 10)
        pdf.multi_cell(0, 5, f"Generado: {datetime.now():%d-%m-%Y %H:%M} | Sorteos analizados: {R['n_sorteos']} ({JUEGO_ANALISIS}) | "
                             f"Proximo sorteo: {DIAS_SORTEO[ps.weekday()]} {ps:%d-%m-%Y}", new_x="LMARGIN", new_y="NEXT")
        pdf.ln(3); pdf.set_font("Helvetica", "B", 12); pdf.cell(0, 8, "Tickets sugeridos", new_x="LMARGIN", new_y="NEXT"); pdf.set_font("Courier", "", 10)
        for t in tickets:
            pdf.cell(0, 6, f"Ticket {t['id']}: {' - '.join(f'{x:02d}' for x in t['balotas'])}  + SB {t['sb']:02d}   (suma {t['suma']}, {t['pares']} pares, unicidad {t['unicidad']:.0f})", new_x="LMARGIN", new_y="NEXT")
        pdf.ln(3); pdf.set_font("Helvetica", "B", 12); pdf.cell(0, 8, "Resumen analitico", new_x="LMARGIN", new_y="NEXT"); pdf.set_font("Helvetica", "", 10)
        for linea in [f"Calientes (ult. {VENTANA_RECIENTE}): {R['calientes_recientes']}", f"Frios (ult. {VENTANA_RECIENTE}): {R['frios_recientes']}",
                      f"Pares por sorteo: media {R['pares_media']:.2f} (teorico 2.44)", f"Suma: media {R['suma_media']:.1f}, 80% entre {R['suma_p10']:.0f} y {R['suma_p90']:.0f}",
                      f"Superbalota: media {R['sb_media']:.2f}, varianza {R['sb_var']:.2f} (teorica {R['sb_var_teorica']:.2f})",
                      "Probabilidad del premio mayor: 1 en 15.401.568 para cualquier combinacion.",
                      *( [f"Valor esperado por tiquete Baloto: {fmt_cop(ev_baloto.Aporte_EV.sum())} de {fmt_cop(PRECIO_BALOTO)}"] if "ev_baloto" in globals() else [])]:
            pdf.multi_cell(0, 5, "- " + linea, new_x="LMARGIN", new_y="NEXT")
        img = DATA_DIR / "panel_estadistico.png"
        if img.exists():
            pdf.add_page(); pdf.image(str(img), x=10, w=190)
        pdfp = DATA_DIR / f"{nombre}.pdf"; pdf.output(str(pdfp)); salidas.append(pdfp)
    except ImportError:
        print("ℹ️ fpdf2 no instalado → solo Excel. (En Colab la celda 1 lo instala.)")
    for s in salidas:
        print(f"📤 {s}  ({s.stat().st_size/1024:.1f} KB)")
    if EN_COLAB:
        from google.colab import files
        for s in salidas:
            files.download(str(s))
    return salidas

exportar_informe()


## 📚 Notas metodológicas y mantenimiento

**Fuente de datos.** `https://baloto.com/resultados-baloto/<N>` y `/resultados-revancha/<N>` (N = número de sorteo). La página oficial publica detalle desde ~#2.076 (jul-2021). El CSV antiguo (2009-2026) tiene fechas/numeración que **no coinciden** con la web oficial, por eso la v6 no lo mezcla.

**Prueba χ² de uniformidad.** Compara la frecuencia observada de cada balota con la esperada (n·5/43). Un *p* > 0,05 significa que las diferencias (los "calientes" y "fríos") son del tamaño que produce el puro azar.

**Validación con ganadores reales.** Si los sorteos son uniformes, la proporción de ganadores por categoría debe seguir la combinatoria. La razón observado/esperado ≈ 1 lo confirma con millones de tiquetes reales: ningún patrón histórico altera las probabilidades.

**Cambio de reglas detectado en los datos (#2192, 25-may-2022).** Antes de ese sorteo la categoría "0 + SB" pagaba solo con 0 balotas + superbalota (razón ganadores ≈ 2,5 frente a las demás categorías); desde entonces incluye 1 balota + SB (razón ≈ 4,4). Coincide con la entrada del operador actual (Contrato C1855 de 2022, Acuerdo 01 de 2022). El notebook usa el régimen correcto para cada sorteo. El premio mínimo pasó de $5.700 a $6.000 en 2025, señal de que el precio del tiquete cambió: verifica `PRECIO_BALOTO`.

**Valor esperado (EV).** Σ (probabilidad de categoría × premio neto típico). Premios menores = mediana oficial de los últimos 40 sorteos; premio mayor = acumulado actual menos 20 % de ganancia ocasional. El EV siempre está muy por debajo del precio: Baloto es entretenimiento, no inversión.

**Motor v6.** Pesos = 0,4·frecuencia reciente + 0,2·rezago + 0,4·uniforme (ajustables). Filtros de forma evitan combinaciones populares (todas ≤31, consecutivas, misma terminación, ganadores pasados). El **índice de unicidad** reemplaza la "confianza" ficticia de la v5.

**Backtesting.** Walk-forward: el motor solo ve el pasado de cada sorteo evaluado y se compara contra tickets al azar con pruebas de Mann-Whitney y χ². Resultado esperado: sin diferencia significativa.

**Si baloto.com cambia su HTML**, ajusta `_RE_CABECERA` y `_RE_PREMIO` en la celda 2; el resto no depende del formato.

**Precio del tiquete y UVT** son constantes en la celda 1: verifícalas cada año.

---
*Baloto Master AI v6.0 · noviembre 2026 · Juega con responsabilidad (Ley 643 de 2001, Coljuegos).*
